# Test Set Segmentation Model Comparison

Compare the naive baseline, classical ML baseline, and final YOLO11n-seg model on the held-out test set.

This notebook produces:
- a report-ready metrics table for mIoU, Dice/F1, precision, recall, and boundary F1
- per-image metrics for error analysis
- visual examples with source image, expected mask, predicted mask, and error overlay
- focused panels for the highest-error samples

## 1. Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether the notebook is launched from repo root or notebooks/."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "gunpla_studio" / "segmenters.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing gunpla_studio/segmenters.py")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from gunpla_studio.segmenters import ClassicalMLSegmenter, NaiveBaselineSegmenter, YOLOSegSegmenter

DATASET_ROOT = PROJECT_ROOT / "data" / "processed" / "gunpla-yolov7"
TEST_IMAGE_DIR = DATASET_ROOT / "test" / "images"
TEST_LABEL_DIR = DATASET_ROOT / "test" / "labels"
YOLO_MODEL_PATH = PROJECT_ROOT / "models" / "gunpla_yolo11n_seg.pt"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs" / "test_set_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to a small integer while iterating quickly. Use None for the full test set.
MAX_TEST_IMAGES: int | None = None

# Boundary F1 tolerance in pixels. A small tolerance avoids over-penalizing 1-2 px contour shifts.
BOUNDARY_TOLERANCE_PX = 3

print(f"Project root: {PROJECT_ROOT}")
print(f"Test images: {TEST_IMAGE_DIR}")
print(f"YOLO model: {YOLO_MODEL_PATH}")

Project root: C:\Users\developer\projects\aipi540-gunpla-studio
Test images: C:\Users\developer\projects\aipi540-gunpla-studio\data\processed\gunpla-yolov7\test\images
YOLO model: C:\Users\developer\projects\aipi540-gunpla-studio\models\gunpla_yolo11n_seg.pt


## 2. Load Test Set and Ground Truth Masks

In [2]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_test_images() -> list[Path]:
    image_paths = sorted(path for path in TEST_IMAGE_DIR.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    if MAX_TEST_IMAGES is not None:
        image_paths = image_paths[:MAX_TEST_IMAGES]
    return image_paths


def yolo_polygon_label_to_mask(label_path: Path, width: int, height: int) -> np.ndarray:
    """Convert YOLO segmentation polygon labels into one binary foreground mask."""
    mask = np.zeros((height, width), dtype=np.uint8)
    if not label_path.exists():
        return mask.astype(bool)

    for raw_line in label_path.read_text(encoding="utf-8").splitlines():
        parts = raw_line.strip().split()
        if len(parts) < 7:
            continue
        coords = np.array([float(value) for value in parts[1:]], dtype=np.float32)
        if len(coords) % 2 != 0:
            coords = coords[:-1]
        points = coords.reshape(-1, 2)
        points[:, 0] = np.clip(points[:, 0] * width, 0, width - 1)
        points[:, 1] = np.clip(points[:, 1] * height, 0, height - 1)
        cv2.fillPoly(mask, [points.astype(np.int32)], 1)
    return mask.astype(bool)


def load_sample(image_path: Path) -> dict:
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"
    expected_mask = yolo_polygon_label_to_mask(label_path, width, height)
    return {
        "image_id": image_path.stem,
        "image_path": image_path,
        "label_path": label_path,
        "image": image,
        "expected_mask": expected_mask,
    }


test_image_paths = list_test_images()
samples = [load_sample(path) for path in test_image_paths]

print(f"Loaded {len(samples)} test samples")
pd.DataFrame(
    {
        "image_id": [sample["image_id"] for sample in samples],
        "has_label": [sample["label_path"].exists() for sample in samples],
        "foreground_pixels": [int(sample["expected_mask"].sum()) for sample in samples],
    }
).head()

Loaded 20 test samples


,image_id,has_label,foreground_pixels
0,gallery_20201108_171431_png.rf.e51eb859e9f0bbf...,True,163456
1,gallery_Action-1-DSC_1285_png.rf.03ddf41d0189a...,True,116299
2,image_jpg.rf.0868c7ce684a5e17c5a63489721ae74b,True,2231212
3,image_jpg.rf.245b065a4e215f1173583bbd0613e0ef,True,2574445
4,image_jpg.rf.27999b3e6111b9d84e30d324577227cd,True,3676357


## 3. Metrics

In [3]:
def as_binary_mask(mask: np.ndarray) -> np.ndarray:
    return np.asarray(mask) > 0.5


def boundary_mask(mask: np.ndarray) -> np.ndarray:
    mask_u8 = as_binary_mask(mask).astype(np.uint8)
    if mask_u8.sum() == 0:
        return np.zeros_like(mask_u8, dtype=bool)
    kernel = np.ones((3, 3), np.uint8)
    eroded = cv2.erode(mask_u8, kernel, iterations=1)
    return (mask_u8 - eroded).astype(bool)


def boundary_f1_score(expected: np.ndarray, predicted: np.ndarray, tolerance_px: int = 3) -> float:
    expected_boundary = boundary_mask(expected)
    predicted_boundary = boundary_mask(predicted)

    if expected_boundary.sum() == 0 and predicted_boundary.sum() == 0:
        return 1.0
    if expected_boundary.sum() == 0 or predicted_boundary.sum() == 0:
        return 0.0

    kernel_size = max(1, tolerance_px * 2 + 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    expected_dilated = cv2.dilate(expected_boundary.astype(np.uint8), kernel).astype(bool)
    predicted_dilated = cv2.dilate(predicted_boundary.astype(np.uint8), kernel).astype(bool)

    precision = (predicted_boundary & expected_dilated).sum() / max(predicted_boundary.sum(), 1)
    recall = (expected_boundary & predicted_dilated).sum() / max(expected_boundary.sum(), 1)
    return float(2 * precision * recall / max(precision + recall, 1e-9))


def segmentation_metrics(expected: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    expected = as_binary_mask(expected)
    predicted = as_binary_mask(predicted)

    tp = int((expected & predicted).sum())
    fp = int((~expected & predicted).sum())
    fn = int((expected & ~predicted).sum())
    tn = int((~expected & ~predicted).sum())

    iou = tp / max(tp + fp + fn, 1)
    dice_f1 = (2 * tp) / max(2 * tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    return {
        "mIoU": float(iou),
        "Dice/F1": float(dice_f1),
        "Precision": float(precision),
        "Recall": float(recall),
        "Boundary F1": boundary_f1_score(expected, predicted, BOUNDARY_TOLERANCE_PX),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


## 4. Run the Three Segmenters

In [ ]:
segmenters = {
    "Naive baseline": NaiveBaselineSegmenter(),
    "Classical ML (GrabCut)": ClassicalMLSegmenter(),
    "YOLO11n-seg final": YOLOSegSegmenter(YOLO_MODEL_PATH),
}

rows: list[dict] = []
predictions: dict[tuple[str, str], np.ndarray] = {}

for sample_index, sample in enumerate(samples, start=1):
    image = sample["image"]
    expected_mask = sample["expected_mask"]
    print(f"[{sample_index}/{len(samples)}] {sample['image_id']}")

    for model_name, segmenter in segmenters.items():
        result = segmenter.segment(image)
        predicted_mask = as_binary_mask(result.mask)
        predictions[(sample["image_id"], model_name)] = predicted_mask

        metric_values = segmentation_metrics(expected_mask, predicted_mask)
        rows.append(
            {
                "image_id": sample["image_id"],
                "image_path": str(sample["image_path"].relative_to(PROJECT_ROOT)),
                "model": model_name,
                "message": result.message,
                **metric_values,
                "error_score": 1.0 - metric_values["Dice/F1"],
            }
        )

per_image_results = pd.DataFrame(rows)
per_image_results.to_csv(OUTPUT_DIR / "per_image_test_metrics.csv", index=False)
per_image_results.head()

[1/20] gallery_20201108_171431_png.rf.e51eb859e9f0bbf7bbdcb310886b13c8
[2/20] gallery_Action-1-DSC_1285_png.rf.03ddf41d0189a163f9e5d32927b84574
[3/20] image_jpg.rf.0868c7ce684a5e17c5a63489721ae74b


## 5. Report-Ready Result Table

In [ ]:
metric_columns = ["mIoU", "Dice/F1", "Precision", "Recall", "Boundary F1"]

summary_table = (
    per_image_results.groupby("model", sort=False)[metric_columns]
    .agg(["mean", "std"])
    .round(4)
)

report_table = per_image_results.groupby("model", sort=False)[metric_columns].mean().round(4).reset_index()
report_table.to_csv(OUTPUT_DIR / "summary_test_metrics.csv", index=False)

display(report_table)
display(summary_table)


In [ ]:
ax = report_table.set_index("model")[metric_columns].plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
ax.set_title("Test Set Segmentation Metrics")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "summary_test_metrics.png", dpi=160)
plt.show()

## 6. Qualitative Error Analysis

In [ ]:
def mask_overlay(image: Image.Image, mask: np.ndarray, color: tuple[int, int, int], alpha: float = 0.45) -> np.ndarray:
    rgb = np.array(image.convert("RGB"), dtype=np.float32)
    overlay = rgb.copy()
    mask_bool = as_binary_mask(mask)
    overlay[mask_bool] = (1 - alpha) * overlay[mask_bool] + alpha * np.array(color, dtype=np.float32)
    return np.clip(overlay, 0, 255).astype(np.uint8)


def error_overlay(image: Image.Image, expected: np.ndarray, predicted: np.ndarray) -> np.ndarray:
    rgb = np.array(image.convert("RGB"), dtype=np.float32)
    expected = as_binary_mask(expected)
    predicted = as_binary_mask(predicted)
    true_positive = expected & predicted
    false_positive = ~expected & predicted
    false_negative = expected & ~predicted

    overlay = rgb.copy()
    overlay[true_positive] = 0.45 * overlay[true_positive] + 0.55 * np.array([46, 204, 113])
    overlay[false_positive] = 0.35 * overlay[false_positive] + 0.65 * np.array([231, 76, 60])
    overlay[false_negative] = 0.35 * overlay[false_negative] + 0.65 * np.array([52, 152, 219])
    return np.clip(overlay, 0, 255).astype(np.uint8)


def get_sample(image_id: str) -> dict:
    for sample in samples:
        if sample["image_id"] == image_id:
            return sample
    raise KeyError(image_id)


def plot_prediction_panel(image_id: str, model_name: str, save_path: Path | None = None) -> None:
    sample = get_sample(image_id)
    image = sample["image"]
    expected = sample["expected_mask"]
    predicted = predictions[(image_id, model_name)]
    metric_row = per_image_results[(per_image_results["image_id"] == image_id) & (per_image_results["model"] == model_name)].iloc[0]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(image)
    axes[0].set_title("Image")
    axes[1].imshow(mask_overlay(image, expected, (52, 152, 219)))
    axes[1].set_title("Expected mask")
    axes[2].imshow(mask_overlay(image, predicted, (231, 76, 60)))
    axes[2].set_title("Predicted mask")
    axes[3].imshow(error_overlay(image, expected, predicted))
    axes[3].set_title("Error overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{model_name} | {image_id}\n"
        f"mIoU={metric_row['mIoU']:.3f}, Dice/F1={metric_row['Dice/F1']:.3f}, "
        f"Precision={metric_row['Precision']:.3f}, Recall={metric_row['Recall']:.3f}, "
        f"Boundary F1={metric_row['Boundary F1']:.3f}",
        y=1.06,
    )
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()


### Best and Worst YOLO11n-seg Examples

In [ ]:
FINAL_MODEL_NAME = "YOLO11n-seg final"
final_model_results = per_image_results[per_image_results["model"] == FINAL_MODEL_NAME].copy()

best_examples = final_model_results.sort_values("Dice/F1", ascending=False).head(3)
worst_examples = final_model_results.sort_values("Dice/F1", ascending=True).head(3)

display(best_examples[["image_id", *metric_columns, "error_score"]])
display(worst_examples[["image_id", *metric_columns, "error_score"]])

In [ ]:
for image_id in best_examples["image_id"]:
    plot_prediction_panel(
        image_id,
        FINAL_MODEL_NAME,
        OUTPUT_DIR / "examples" / f"best_{image_id}.png",
    )

In [ ]:
for image_id in worst_examples["image_id"]:
    plot_prediction_panel(
        image_id,
        FINAL_MODEL_NAME,
        OUTPUT_DIR / "examples" / f"worst_{image_id}.png",
    )

### Highest-Error Samples Across All Methods

In [ ]:
high_error = per_image_results.sort_values("error_score", ascending=False).head(9)
display(high_error[["image_id", "model", *metric_columns, "error_score", "message"]])

for row in high_error.itertuples(index=False):
    safe_model_name = row.model.lower().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
    plot_prediction_panel(
        row.image_id,
        row.model,
        OUTPUT_DIR / "examples" / f"high_error_{safe_model_name}_{row.image_id}.png",
    )

## 7. Report Notes

In [ ]:
winner_by_metric = {
    metric: report_table.loc[report_table[metric].idxmax(), "model"]
    for metric in metric_columns
}

print("Report-ready artifacts")
print(f"- Summary table: {OUTPUT_DIR / 'summary_test_metrics.csv'}")
print(f"- Per-image metrics: {OUTPUT_DIR / 'per_image_test_metrics.csv'}")
print(f"- Metric chart: {OUTPUT_DIR / 'summary_test_metrics.png'}")
print(f"- Example figures: {OUTPUT_DIR / 'examples'}")
print("\nBest model by metric:")
for metric, model_name in winner_by_metric.items():
    print(f"- {metric}: {model_name}")
